<a href="https://colab.research.google.com/github/BWMIN-Hub/SR_practice/blob/main/notebooks/19_scene_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# One scene, end to end

Pages 13 - 18 took the steps one at a time.  Here they run in order on one THEOS-2
scene, and the run ends with a report that someone else can read without the notebook.

| step | what happens |
|---|---|
| 1  prepare | read the header, pick the conversion window from a decimated read |
| 2  infer + write | one block at a time: read, convert, infer in tiles, convert back, write |
| 3  QC | header, geometry, radiometry, seams — each check pass or CHECK |
| 4  IQA | no-reference scores against bicubic, and a consistency check against the input |
| 5  report | one HTML file and one JSON with everything above |

The network is the HAT x3 from page 07, the latest weights (v4) — trained to take
Sentinel-2 at 10 m to 3.3 m.

The scene is the Bangkok crop from page 13: 2434 x 2484 px, four bands, 12-bit in
`uint16`, 0.5 m.  A full THEOS-2 strip is about 26 000 px a side and 5 GB, which does not
fit in Colab — but nothing below ever holds the whole scene, so the same calls run on one.

In [ ]:
import sys, urllib.request

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
!pip install -q timm einops
for f in ('lib/infer.py', 'lib/iqa.py', 'lib/hat_arch.py', 'lib/hat_models.py', 'lib/qcpipe.py'):
    urllib.request.urlretrieve(f'{BASE}/{f}', f.split('/')[-1])
for m in ('infer', 'iqa', 'hat_arch', 'hat_models', 'qcpipe'):
    sys.modules.pop(m, None)

import os
from IPython.display import HTML
from infer import Config
from qcpipe import *
from hat_models import load_hat

get = lambda p, out: urllib.request.urlretrieve(f'{BASE}/{p}', out)[0]
PATH = get('dataset/intake/input.tif', 'scene.tif')
WEIGHT = get('models/07_hat_x3/checkpoints/hat_x3_v4.pth', 'hat_x3_v4.pth')
os.makedirs('iqa', exist_ok=True)
for f in ('niqe_model.npz', 'brisque_svr.npz'):
    get(f'results/iqa/{f}', f'iqa/{f}')

## 1. Prepare

The model was trained on 8-bit images, so the 12-bit values have to be squeezed into
0 - 255 and stretched back afterwards (page 18).  Two decisions:

**One window for the whole scene.**  If every block picked its own, neighbouring blocks
would come out in different brightness.  On a 26 000 px scene the percentiles cannot
come from the full array, so `scan` reads the file decimated to about 1 000 px a side —
plenty for a histogram.

**The brightness the model was trained on.**  The Sentinel-2 patches HAT learned from
have a median around 100 of 255.  Through a plain 2 - 99.5 % window this city scene has
a median of 30: bright roofs stretch the window and push everything else into the dark.
A power curve moves the median to 0.4 (about 100) and is undone exactly on the way back.

In [ ]:
s = scan(PATH, bands=(1, 2, 3), lo=2, hi=99.5, median=0.4)
for b, lo, hi in zip(('red', 'green', 'blue'), s['lo'], s['hi']):
    print(f'{b:6} window {lo:7.1f} - {hi:7.1f}')
print(f'power  {s["gamma"]}')
show_scan(s)

Near infrared is left out: the network takes three channels.  The output will have
three bands, and the checks below expect three.

## 2. Tiled inference, windowed write

Two levels of cutting.

| level | size | why |
|---|---|---|
| block | 768 px | what is read from disk and written back — bounds the **RAM** |
| tile | 192 px | what goes through the network — bounds the **GPU memory** (page 14) |

Each tile reads 32 px around itself, so 256 px go through HAT — a multiple of its 16 px
attention window.  Each block is read with the same margin, so block edges get real
neighbours just as tile edges do, and the tiles start at the block corner, so every tile
boundary in the scene lands on a multiple of 192.

The output file is created once, empty, at its final size and georeferencing; each
block is written straight into its window.

In [ ]:
BLOCK = 768                       # a multiple of the tile
cfg = Config(scale=3, tile=192, overlap=32, batch=4, precision='fp16')
net = load_hat(WEIGHT, scale=3)
show_blocks(s, BLOCK, cfg)

In [ ]:
st = run(PATH, 'sr.tif', net, cfg, s, block=BLOCK, model='HAT x3 (v4)', value_range=1.0)

`value_range=1.0` because HAT takes 0 - 1, not 0 - 255.  `fp16` halves the time here; the
result differs from `fp32` by 0.03 levels on average.

The memory line is the point of the windowed write.  One block costs about 50 MB
whatever the scene size; holding this scene and its result at once would cost 500 MB.
For a 26 000 px THEOS-2 strip that second number becomes about 57 GB, which no Colab
session has, while the first stays where it is.

The file also records how it was made — model, scale, source, window and power curve
are GeoTIFF tags, so the conversion can be undone later without this notebook.

In [ ]:
!gdalinfo sr.tif | grep -E 'Size is|Pixel Size|Origin|SR_|Band [0-9]'

## 3. QC — header, geometry, radiometry

Every check is a row with what was expected and what came out.

| group | asks |
|---|---|
| header | size x3, three bands, same data type and nodata, tiled and compressed, processing tags present |
| geometry | same CRS, pixel size divided by 3, same origin, same footprint, no rotation |
| radiometry | mean within 2 %, std x 0.9 - 1.3, values in range, histogram shape inside the window, how much sits at the window ends, no new nodata |
| seams | the step across tile and block boundaries, against the step a few rows away |

All statistics are accumulated strip by strip, so this too runs on a full scene.

In [ ]:
checks = qc(PATH, 'sr.tif', s, cfg, block=BLOCK)
show_qc(checks)

30 of 33 pass.  Header, geometry, nodata and seams are clean: the step across a tile
or block boundary is 1.12 times the step a few rows away — measurable, not visible.

The three CHECKs are all the same finding: the standard deviation came out at 0.84 -
0.89 of the input while the mean stayed within 2 %.  The output has **less contrast**
than the scene that went in.  That is not the pipeline — the conversion is undone
exactly — it is the model.  HAT was trained to turn Sentinel-2 colours into IKONOS ones
(pages 07 and 09), and it applies the same tone change to anything it is given.

The range row passes, but red comes out 209 - 1058: exactly the window.  The input went
up to 4095.  Whatever lay outside the window was flattened before the network saw it,
and converting back cannot bring it back (page 15).

## 4. IQA

There is no reference at 0.17 m for this scene, so PSNR and SSIM against a ground truth
are not possible.  Two things are:

- **no-reference scores** — NIQE, BRISQUE, PIQE on SR and on bicubic, lower is better.
  They are calibrated on this project's IKONOS images (page 11), so only the comparison
  between the two means anything.
- **consistency** — shrink the result back to 0.5 m and compare it with the input.  A
  result that changed the scene, rather than only adding detail, does not shrink back
  to what went in.

Three crops are picked automatically: the busiest, a typical and the flattest place.

In [ ]:
res = iqa_scan(PATH, 'sr.tif', s, cfg, iqa_dir='iqa')
checks += iqa_rows(res)
show_iqa(res)

Shrunk back to 0.5 m, bicubic matches the input at 41 - 43 dB.  The SR result matches it
at 25 - 30 dB: it is not the same scene any more.  The crops show what changed: lighter
and flatter — the contrast loss the std rows found — and orange fringes along roof edges
that are not in the input.

There is a second mismatch underneath.  HAT learned that one input pixel is 10 m of
ground.  Here one pixel is 0.5 m, so the structures it was taught to sharpen — field
edges, blocks of buildings — are twenty times larger than anything it is shown.

The no-reference scores disagree with each other: NIQE prefers SR on all three crops,
BRISQUE on two, PIQE on none.  Without a reference they can only say "looks different",
which is why the consistency number is the one that becomes a check.

## 5. The report

One call writes `qc_report.html` — self-contained, every figure embedded — and
`qc_report.json` with the same numbers for a script to read.  The verdict at the top is
PASS only if every check passed.

In [ ]:
html = write_report('qc_report.html', PATH, 'sr.tif', s, st, checks, res)
HTML(html)

The verdict is CHECK, 5 of 36: three std rows and two consistency rows, all saying the
same thing.  Everything the pipeline controls — size, georeferencing, tags, nodata,
seams — passed.  What failed is the model's fit to this sensor.

That is what the report is for.  Whoever decides whether this file goes out sees, on one
page, that it is geometrically correct and radiometrically not faithful to the input —
and does not have to open a notebook to find out.

In Colab the files are in the panel on the left;
`from google.colab import files; files.download('qc_report.html')` saves one.

## 6. When the output is wrong

Two copies of the result, each with a classic mistake.  The report only earns its place
if it catches them.

| bug | what was skipped |
|---|---|
| `no_rescale` | the input's transform was copied without dividing the pixel size |
| `stay_8bit` | the network's 0 - 255 output was written without converting back |

In [ ]:
show_qc(qc(PATH, break_output('sr.tif', 'bad1.tif', 'no_rescale'), s, cfg, block=BLOCK),
        only_bad=True)

The pixels are identical to the real result, so radiometry gives the same three std rows
as before.  The new failures are geometry: the pixel size is still 0.5 m, so the footprint
is three times as wide and tall — the bounds are 2.5 km off, and in GIS the image would
cover nine times the ground, over the wrong streets.

In [ ]:
show_qc(qc(PATH, break_output('sr.tif', 'bad2.tif', 'stay_8bit'), s, cfg, block=BLOCK),
        only_bad=True)

Geometry is perfect and radiometry fails everywhere: the mean dropped by about 70 %, the values
stop at 255, and 950 000 pixels that left the network as 0 now read as nodata.
Opened on its own with an automatic stretch, this file looks fine.